# AgentForge SWE-smith training collection (50 shards)

Run this notebook from an interactive CX3 GPU allocation. It collects the pinned 5,000-task SWE-smith training split as 50 deterministic shards, verifies the complete collection, and evaluates all eight sample slots.

Normal workflow: run setup and vLLM once, set `SHARD_INDEX`, run the collection block once for each index `0..49`, run the strict verification block, then enable evaluation. Collection is resumable: rerunning a shard retains completed/model-terminated outcomes and retries unexpected failures.

## 1. Repository, environment, and run configuration

Use a stable `RUN_NAME` for all 50 shards. Change only `SHARD_INDEX` between collection runs. The default split and revision match `cluster/submit_swesmith_full.sh`.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').is_file(), f'Open this notebook from debug-depo, not {ROOT}'

SCRATCH = Path(os.environ.get('DEBUG_DEPO_SCRATCH', ROOT / 'scratch')).expanduser()
RUN_NAME = os.environ.get('RUN_NAME', 'swesmith-train-5000')
RUN_ROOT = Path(os.environ.get('RUN_ROOT', SCRATCH / 'runs' / RUN_NAME)).expanduser()

DATASET = 'SWE-bench/SWE-smith-py'
DATASET_REVISION = '77cab9055d42ab4a5c25c89a8f937096db13558e'
SPLIT = 'train'
TASK_IDS_FILE = ROOT / 'data/splits/swesmith_train_5000_instance_ids.txt'
EXPECTED_TASKS = 5_000
NUM_SHARDS = 50
SHARD_INDEX = 0  # EDIT THIS: an integer from 0 through 49

TEMPERATURES = '0.6:0.7'
RUNS_PER_TEMPERATURE = 4
TOTAL_SAMPLES = 8
BASE_SEED = 42
ROLLOUT_WORKERS = 6
EVAL_MAX_WORKERS = 25

AGENTFORGE_MODEL = 'Kwai-Klear/Klear-AgentForge-8B-SFT'
MINI_SWE_MODEL = f'hosted_vllm/{AGENTFORGE_MODEL}'
VLLM_HOST = '127.0.0.1'
VLLM_PORT = 8000
LLM_BASE_URL = f'http://{VLLM_HOST}:{VLLM_PORT}/v1'

assert 0 <= SHARD_INDEX < NUM_SHARDS
assert TASK_IDS_FILE.is_file(), TASK_IDS_FILE
selected_ids = [line.strip() for line in TASK_IDS_FILE.read_text().splitlines() if line.strip()]
assert len(selected_ids) == EXPECTED_TASKS, (len(selected_ids), EXPECTED_TASKS)
assert len(set(selected_ids)) == EXPECTED_TASKS, 'The tracked task list contains duplicates.'

print(json.dumps({
    'repository': str(ROOT),
    'run_root': str(RUN_ROOT),
    'shard_index': SHARD_INDEX,
    'num_shards': NUM_SHARDS,
    'tasks': EXPECTED_TASKS,
    'expected_tasks_this_shard': EXPECTED_TASKS // NUM_SHARDS,
    'rollouts_this_shard': (EXPECTED_TASKS // NUM_SHARDS) * TOTAL_SAMPLES,
}, indent=2))

## 2. Setup/preflight

The kernel should be the environment created by `bash cluster/setup_jupyter_env.sh`. This cell checks the Python packages, Apptainer, GPU access, vLLM image, and shared cache paths before doing expensive work.

In [ ]:
env = os.environ.copy()
env.update({
    'DEBUG_DEPO_ROOT': str(ROOT),
    'DEBUG_DEPO_SCRATCH': str(SCRATCH),
    'RUN_NAME': RUN_NAME,
    'RUN_ROOT': str(RUN_ROOT),
    'PYTHONPATH': os.pathsep.join(filter(None, [
        str(ROOT / 'src'),
        str(ROOT / 'external/mini-swe-agent-plus/src'),
        str(ROOT / 'external/SWE-smith'),
        env.get('PYTHONPATH', ''),
    ])),
})

RUN_ROOT.mkdir(parents=True, exist_ok=True)
checks = {
    'python': sys.executable,
    'apptainer': shutil.which('apptainer'),
    'nvidia-smi': shutil.which('nvidia-smi'),
    'curl': shutil.which('curl'),
    'vllm_image': str(Path(env.get('VLLM_IMAGE', ROOT / 'cluster/apptainer/vllm-openai.sif'))),
}
print(json.dumps(checks, indent=2))
subprocess.run([
    sys.executable, '-c',
    'import debug_depo, minisweagent, swesmith; print("Python packages: OK")',
], cwd=ROOT, env=env, check=True)
for executable in ('apptainer', 'nvidia-smi', 'curl'):
    assert checks[executable], f'Missing {executable}'
assert Path(checks['vllm_image']).is_file(), f"Missing vLLM image: {checks['vllm_image']}"
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], check=True)

## 3. Start vLLM

Set `START_VLLM = True` once per kernel session. The process writes to the run root and remains available while later cells collect shards. If a server is already running at `LLM_BASE_URL`, leave it false.

In [ ]:
START_VLLM = False
VLLM_STARTUP_TIMEOUT = 7200
vllm_process = globals().get('vllm_process')
vllm_log_handle = globals().get('vllm_log_handle')

def url_json(url, *, method='GET', payload=None, timeout=30):
    body = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(url, data=body, method=method)
    request.add_header('Content-Type', 'application/json')
    request.add_header('Authorization', 'Bearer local')
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)

if START_VLLM:
    if vllm_process is not None and vllm_process.poll() is None:
        print(f'vLLM is already owned by this kernel (PID {vllm_process.pid}).')
    else:
        vllm_env = env.copy()
        vllm_env.update({
            'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
            'MINI_SWE_MODEL': MINI_SWE_MODEL,
            'HOST': VLLM_HOST,
            'PORT': str(VLLM_PORT),
            'CONTEXT_LENGTH': '65536',
        })
        vllm_log_path = RUN_ROOT / 'vllm-notebook.log'
        vllm_log_handle = vllm_log_path.open('a', encoding='utf-8')
        vllm_process = subprocess.Popen(
            ['bash', 'cluster/apptainer/serve_vllm.sh'],
            cwd=ROOT, env=vllm_env,
            stdout=vllm_log_handle, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        deadline = time.monotonic() + VLLM_STARTUP_TIMEOUT
        while True:
            if vllm_process.poll() is not None:
                raise RuntimeError(f'vLLM exited with {vllm_process.returncode}; inspect {vllm_log_path}')
            try:
                models = url_json(f'{LLM_BASE_URL}/models', timeout=2)
                print(f'vLLM ready (PID {vllm_process.pid}): {models}')
                break
            except (OSError, urllib.error.URLError, TimeoutError):
                if time.monotonic() >= deadline:
                    raise TimeoutError(f'vLLM did not become ready; inspect {vllm_log_path}')
                time.sleep(5)
else:
    print('START_VLLM is false; expecting an existing server at', LLM_BASE_URL)

## 4. Test vLLM

This checks both model discovery and a real OpenAI-compatible chat completion before collection.

In [ ]:
models_payload = url_json(f'{LLM_BASE_URL}/models', timeout=10)
served_models = [item['id'] for item in models_payload.get('data', [])]
assert served_models, models_payload
print('Served models:', served_models)

completion = url_json(
    f'{LLM_BASE_URL}/chat/completions',
    method='POST',
    payload={
        'model': served_models[0],
        'messages': [{'role': 'user', 'content': 'Reply with exactly: READY'}],
        'temperature': 0,
        'max_tokens': 16,
    },
    timeout=120,
)
assert completion.get('choices'), completion
print(completion['choices'][0]['message']['content'])

## 5. Collect one selected shard

Return to the configuration cell, choose one `SHARD_INDEX` from 0 to 49, then run this block. Each shard contains 100 tasks × 8 samples = 800 trajectories. `scripts/collect_swesmith.sh` uses strict completion; expected model terminations are valid outcomes, while unexpected execution/infra errors make the cell fail. Rerun the same shard to retry failed slots.

In [ ]:
RUN_SELECTED_SHARD = False
OVERWRITE_COMPLETED_SHARD = False  # normally keep False; collection is resumable

def collection_environment(shard_index):
    assert 0 <= shard_index < NUM_SHARDS
    shard_output = RUN_ROOT / 'collection' / f'shard-{shard_index}'
    collection_env = env.copy()
    for key in ('LIMIT', 'INSTANCE_ID', 'START_INDEX', 'OVERWRITE'):
        collection_env.pop(key, None)
    collection_env.update({
        'DATASET': DATASET,
        'SWESMITH_DATASET_REVISION': DATASET_REVISION,
        'SPLIT': SPLIT,
        'TASK_IDS_FILE': str(TASK_IDS_FILE),
        'OUTPUT_DIR': str(shard_output),
        'EXPECTED_TASKS': str(EXPECTED_TASKS),
        'NUM_SHARDS': str(NUM_SHARDS),
        'SHARD_INDEX': str(shard_index),
        'TEMPERATURES': TEMPERATURES,
        'RUNS_PER_TEMPERATURE': str(RUNS_PER_TEMPERATURE),
        'BASE_SEED': str(BASE_SEED),
        'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
        'MINI_SWE_MODEL': MINI_SWE_MODEL,
        'MINI_SWE_RUNNER': 'singularity',
        'MINI_SWE_ENVIRONMENT_CLASS': 'singularity',
        'MSWEA_SINGULARITY_EXECUTABLE': 'apptainer',
        'LLM_BASE_URL': LLM_BASE_URL,
        'LLM_API_KEY': 'local',
        'ROLLOUT_WORKERS': str(ROLLOUT_WORKERS),
        'MINI_SWE_WORKERS': '1',
        'MAX_STEPS': '200',
        'CONTEXT_LENGTH': '65536',
        'TOP_P': '1.0',
        'TIMEOUT_SECONDS': '21600',
        'STREAM_OUTPUT': '0',
    })
    if OVERWRITE_COMPLETED_SHARD:
        collection_env['OVERWRITE'] = '1'
    return shard_output, collection_env

shard_output, shard_env = collection_environment(SHARD_INDEX)
print(json.dumps({
    'enabled': RUN_SELECTED_SHARD,
    'shard': f'{SHARD_INDEX}/{NUM_SHARDS - 1}',
    'output': str(shard_output),
    'overwrite': OVERWRITE_COMPLETED_SHARD,
}, indent=2))

if RUN_SELECTED_SHARD:
    subprocess.run(['bash', 'scripts/collect_swesmith.sh'], cwd=ROOT, env=shard_env, check=True)
    print((shard_output / 'summary.json').read_text())
else:
    print('Preview only. Set RUN_SELECTED_SHARD = True to collect this shard.')

## 6. Verify all 50 shards before evaluation

This is a hard gate. It verifies all manifests, exact task-ID coverage with no duplicates, all 40,000 rollout records, zero error/active/unreadable trajectories, and exactly 5,000 unique predictions in every sample slot. Empty patches and `model_terminated` are valid model outcomes; status `error`, missing artifacts, malformed JSON, duplicates, or inconsistent manifests are not.

In [ ]:
from debug_depo.swesmith_progress import inspect_collection

def read_jsonl(path):
    rows = []
    with Path(path).open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise AssertionError(f'Malformed JSONL at {path}:{line_number}') from exc
                assert isinstance(row, dict), f'Non-object row at {path}:{line_number}'
                rows.append(row)
    return rows

def verify_complete_collection(run_root):
    run_root = Path(run_root).expanduser().resolve()
    progress = inspect_collection(run_root)
    problems = list(progress.warnings)
    if len(progress.shards) != NUM_SHARDS:
        problems.append(f'expected {NUM_SHARDS} shards, inspected {len(progress.shards)}')
    if progress.expected_tasks != EXPECTED_TASKS:
        problems.append(f'manifest expected_tasks={progress.expected_tasks}, expected {EXPECTED_TASKS}')
    if progress.samples_per_task != TOTAL_SAMPLES:
        problems.append(f'manifest samples_per_task={progress.samples_per_task}, expected {TOTAL_SAMPLES}')
    if progress.expected_task_ids != set(selected_ids):
        missing = set(selected_ids) - progress.expected_task_ids
        extra = progress.expected_task_ids - set(selected_ids)
        problems.append(f'manifest task-ID mismatch: missing={len(missing)}, extra={len(extra)}')
    for shard in progress.shards:
        if not shard.manifest_present or shard.state != 'complete':
            problems.append(f'shard-{shard.index}: state={shard.state}, manifest={shard.manifest_present}')
        if shard.error_rollouts or shard.active_rollouts or shard.unreadable_rollouts:
            problems.append(
                f'shard-{shard.index}: errors={shard.error_rollouts}, active={shard.active_rollouts}, '
                f'unreadable={shard.unreadable_rollouts}, reasons={shard.error_reasons}'
            )
        if shard.finished_rollouts != shard.expected_rollouts:
            problems.append(
                f'shard-{shard.index}: finished={shard.finished_rollouts}, expected={shard.expected_rollouts}'
            )
    if progress.collected_tasks != EXPECTED_TASKS:
        problems.append(f'fully collected tasks={progress.collected_tasks}, expected={EXPECTED_TASKS}')
    if progress.finished_rollouts != EXPECTED_TASKS * TOTAL_SAMPLES:
        problems.append(f'finished rollouts={progress.finished_rollouts}, expected={EXPECTED_TASKS * TOTAL_SAMPLES}')

    expected_id_set = set(selected_ids)
    for sample_index in range(TOTAL_SAMPLES):
        paths = sorted(run_root.glob(f'collection/shard-*/samples/sample-{sample_index}/predictions.jsonl'))
        if len(paths) != NUM_SHARDS:
            problems.append(f'sample-{sample_index}: prediction shard files={len(paths)}, expected={NUM_SHARDS}')
            continue
        ids = []
        for path in paths:
            rows = read_jsonl(path)
            ids.extend(str(row.get('instance_id', '')) for row in rows)
        if len(ids) != EXPECTED_TASKS or len(set(ids)) != EXPECTED_TASKS or set(ids) != expected_id_set:
            problems.append(
                f'sample-{sample_index}: rows={len(ids)}, unique={len(set(ids))}, '
                f'missing={len(expected_id_set - set(ids))}, extra={len(set(ids) - expected_id_set)}'
            )

    report = {
        'run_root': str(run_root),
        'shards_complete': sum(s.state == 'complete' for s in progress.shards),
        'collected_tasks': progress.collected_tasks,
        'finished_rollouts': progress.finished_rollouts,
        'error_rollouts': progress.error_rollouts,
        'active_rollouts': progress.active_rollouts,
        'unreadable_rollouts': progress.unreadable_rollouts,
        'problems': problems,
    }
    print(json.dumps(report, indent=2))
    if problems:
        raise AssertionError(f'Collection is NOT safe to evaluate ({len(problems)} problem(s)).')
    print('PASS: all 50 shards are complete and safe to evaluate.')
    return report

collection_verification = verify_complete_collection(RUN_ROOT)

## 7. Evaluate the whole current collection

The runner merges all 50 prediction files separately for each of the eight sample slots, checks for exactly 5,000 predictions, and performs strict Apptainer evaluation. It is resumable through cached reports. The verification function is called again immediately before execution.

In [ ]:
RUN_CURRENT_EVALUATION = False

def evaluation_environment(run_root):
    run_root = Path(run_root).expanduser().resolve()
    evaluation_env = env.copy()
    evaluation_env.update({
        'SWESMITH_MODE': 'full',
        'RUN_NAME': run_root.name,
        'RUN_ROOT': str(run_root),
        'DATASET': DATASET,
        'SWESMITH_DATASET_REVISION': DATASET_REVISION,
        'SPLIT': SPLIT,
        'TASK_IDS_FILE': str(TASK_IDS_FILE),
        'EXPECTED_TASKS': str(EXPECTED_TASKS),
        'NUM_SHARDS': str(NUM_SHARDS),
        'TOTAL_SAMPLES': str(TOTAL_SAMPLES),
        'EVAL_MAX_WORKERS': str(EVAL_MAX_WORKERS),
        'SWESMITH_EVAL_RUNTIME': 'apptainer',
    })
    return evaluation_env

def evaluate_complete_run(run_root):
    run_root = Path(run_root).expanduser().resolve()
    verify_complete_collection(run_root)
    subprocess.run(
        ['bash', 'cluster/run_swesmith_evaluation_job.sh'],
        cwd=ROOT,
        env=evaluation_environment(run_root),
        check=True,
    )
    summaries = sorted(run_root.glob('evaluation/sample-*/summary.json'))
    assert len(summaries) == TOTAL_SAMPLES, f'Expected {TOTAL_SAMPLES} summaries, found {len(summaries)}'
    print(f'PASS: evaluated all {TOTAL_SAMPLES} sample slots under {run_root}')
    return summaries

if RUN_CURRENT_EVALUATION:
    current_evaluation_summaries = evaluate_complete_run(RUN_ROOT)
else:
    print('Evaluation disabled. Set RUN_CURRENT_EVALUATION = True after verification passes.')

## 8. Evaluate collections from previous runs

Add one or more previous `RUN_ROOT` directories below. Every run must use the same pinned 5,000 task IDs, 50-shard layout, and eight-sample schedule; each is independently verified before evaluation. Existing valid reports are reused unless you deliberately set `OVERWRITE=1` in `evaluation_environment`.

In [ ]:
PREVIOUS_RUN_ROOTS = [
    # SCRATCH / 'runs' / 'swesmith-train-previous-run',
]
RUN_PREVIOUS_EVALUATIONS = False

previous_evaluation_results = {}
for previous_root in map(Path, PREVIOUS_RUN_ROOTS):
    print(f'\n=== {previous_root} ===')
    if RUN_PREVIOUS_EVALUATIONS:
        previous_evaluation_results[str(previous_root)] = evaluate_complete_run(previous_root)
    else:
        verify_complete_collection(previous_root)
        print('Verified only; set RUN_PREVIOUS_EVALUATIONS = True to evaluate.')

if not PREVIOUS_RUN_ROOTS:
    print('No previous runs selected. Add paths to PREVIOUS_RUN_ROOTS when needed.')

## 9. Analyse evaluated collections

Analysis combines collection metadata and all eight evaluation summaries into rollout-level and task-level tables, per-temperature pass@k metrics, and mixed-temperature pass@k metrics. It refuses to run unless the collection verification passes and all eight evaluation summaries exist.

In [ ]:
RUN_CURRENT_ANALYSIS = False
RUN_PREVIOUS_ANALYSES = False

def analyse_complete_run(run_root):
    run_root = Path(run_root).expanduser().resolve()
    verify_complete_collection(run_root)
    evaluation_summaries = sorted(run_root.glob('evaluation/sample-*/summary.json'))
    if len(evaluation_summaries) != TOTAL_SAMPLES:
        raise AssertionError(
            f'Analysis requires {TOTAL_SAMPLES} evaluation summaries; found '
            f'{len(evaluation_summaries)} under {run_root}'
        )

    analysis_env = env.copy()
    analysis_env.update({
        'RUN_NAME': run_root.name,
        'RUN_ROOT': str(run_root),
        'RUNS_PER_TEMPERATURE': str(RUNS_PER_TEMPERATURE),
        'TOTAL_SAMPLES': str(TOTAL_SAMPLES),
        'EXPECTED_TASKS': str(EXPECTED_TASKS),
    })
    subprocess.run(
        ['bash', 'scripts/analyze_swesmith.sh'],
        cwd=ROOT,
        env=analysis_env,
        check=True,
    )
    summary_path = run_root / 'analysis' / 'summary.json'
    assert summary_path.is_file(), f'Missing analysis summary: {summary_path}'
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps({
        'run_root': str(run_root),
        'tasks': summary.get('tasks'),
        'rollouts': summary.get('rollouts'),
        'evaluated_rollouts': summary.get('evaluated_rollouts'),
        'resolved_rollouts': summary.get('resolved_rollouts'),
        'fully_evaluated_tasks': summary.get('fully_evaluated_tasks'),
        'temperatures': summary.get('temperatures'),
        'mixed_temperature_pass_at_k': summary.get('mixed_temperature_pass_at_k'),
        'summary_path': str(summary_path),
    }, indent=2))
    return summary

if RUN_CURRENT_ANALYSIS:
    current_analysis = analyse_complete_run(RUN_ROOT)
else:
    print('Current-run analysis disabled. Set RUN_CURRENT_ANALYSIS = True after evaluation.')

previous_analysis_results = {}
if RUN_PREVIOUS_ANALYSES:
    for previous_root in map(Path, PREVIOUS_RUN_ROOTS):
        print(f'\n=== Analysing {previous_root} ===')
        previous_analysis_results[str(previous_root)] = analyse_complete_run(previous_root)
elif PREVIOUS_RUN_ROOTS:
    print('Previous-run analysis disabled. Set RUN_PREVIOUS_ANALYSES = True when ready.')

## Operational notes

- Do not change `RUN_NAME`, task IDs, temperatures, seeds, or shard count between shards.
- The 50 shards are deterministic modulo partitions; their order does not matter.
- If a shard cell fails, inspect `collection/shard-N/samples/*/trajectories/*/stderr.txt` and rerun that same shard with overwrite left false.
- Evaluation and analysis do not require vLLM, so the server may be stopped after collection.
- For unattended execution, `cluster/submit_swesmith_full.sh` submits the equivalent `collect → evaluate → analyse` PBS dependency chain.